# MiniExcel — an ipywidgets Excel alternative

A spreadsheet that runs inside a Jupyter notebook. Features:
- Editable grid with row/column headers (A, B, C / 1, 2, 3)
- Formulas: `=SUM`, `=AVERAGE`, `=MIN`, `=MAX`, `=COUNT`, `=PRODUCT`, `=ABS`, `=ROUND`, `=IF`, `=CONCAT`
- Arithmetic: `+ - * / ^`, parentheses, comparisons, negation
- Cell references (`A1`) and ranges (`A1:B5`)
- Automatic recalculation of dependent cells
- Undo / Redo
- Insert / delete rows and columns
- Load & save CSV / XLSX
- Sample data already loaded so you can play immediately

### Setup
Run this once if you don't already have the dependencies:
```
pip install ipywidgets pandas openpyxl
```
Then run the cells below in order.

In [8]:
# Cell 1: The spreadsheet engine.
# This is the same code as spreadsheet_engine.py but embedded for a self-contained notebook.

import re, csv, copy
from typing import Any

def col_letter(col):
    s, n = '', col
    while True:
        s = chr(ord('A') + n % 26) + s
        n = n // 26 - 1
        if n < 0: break
    return s

def col_index(letters):
    n = 0
    for ch in letters.upper():
        n = n * 26 + (ord(ch) - ord('A') + 1)
    return n - 1

def a1_to_rc(ref):
    m = re.fullmatch(r'([A-Za-z]+)(\d+)', ref.strip())
    if not m: raise ValueError(f'Bad cell reference: {ref}')
    letters, digits = m.groups()
    return int(digits) - 1, col_index(letters)

def rc_to_a1(row, col):
    return f'{col_letter(col)}{row + 1}'

class FormulaError(Exception): pass

def _product(xs):
    p = 1
    for x in xs: p *= x
    return p

FUNCTIONS = {
    'SUM': lambda xs: sum(xs),
    'AVERAGE': lambda xs: (sum(xs)/len(xs)) if xs else FormulaError('#DIV/0!'),
    'MIN': lambda xs: min(xs) if xs else FormulaError('#VALUE!'),
    'MAX': lambda xs: max(xs) if xs else FormulaError('#VALUE!'),
    'COUNT': lambda xs: len(xs),
    'PRODUCT': lambda xs: _product(xs),
    'ABS': lambda xs: abs(xs[0]) if len(xs)==1 else FormulaError('#VALUE!'),
    'ROUND': lambda xs: round(xs[0], int(xs[1])) if len(xs)==2 else FormulaError('#VALUE!'),
}

class Evaluator:
    def __init__(self, get_cell):
        self.get_cell = get_cell
        self.deps = set()
    def evaluate(self, text):
        self.deps = set()
        self.tokens = self._tokenize(text); self.pos = 0
        r = self._expr()
        if self.pos != len(self.tokens): raise FormulaError('#SYNTAX')
        return r
    def _tokenize(self, s):
        tokens, i = [], 0
        while i < len(s):
            c = s[i]
            if c.isspace(): i += 1
            elif c.isdigit() or (c=='.' and i+1<len(s) and s[i+1].isdigit()):
                j = i
                while j < len(s) and (s[j].isdigit() or s[j]=='.'): j += 1
                tokens.append(('NUM', float(s[i:j]))); i = j
            elif c == '"':
                j = i + 1
                while j < len(s) and s[j] != '"': j += 1
                tokens.append(('STR', s[i+1:j])); i = j + 1
            elif c.isalpha():
                j = i
                while j < len(s) and (s[j].isalnum() or s[j]=='_'): j += 1
                w = s[i:j]
                if re.fullmatch(r'[A-Za-z]+\d+', w): tokens.append(('REF', w.upper()))
                elif w.upper() in ('TRUE','FALSE'): tokens.append(('BOOL', w.upper()=='TRUE'))
                else: tokens.append(('FUNC', w.upper()))
                i = j
            elif c in '+-*/^(),:<>=':
                if c in '<>' and i+1<len(s) and s[i+1]=='=':
                    tokens.append(('OP', c+'=')); i += 2
                elif c == '<' and i+1<len(s) and s[i+1]=='>':
                    tokens.append(('OP', '<>')); i += 2
                else: tokens.append(('OP', c)); i += 1
            else: raise FormulaError(f"#SYNTAX: unexpected '{c}'")
        return tokens
    def _peek(self, o=0):
        return self.tokens[self.pos+o] if self.pos+o<len(self.tokens) else (None, None)
    def _eat(self):
        t = self.tokens[self.pos]; self.pos += 1; return t
    def _match(self, kind, val=None):
        t = self._peek()
        if t[0] == kind and (val is None or t[1] == val): return self._eat()
        return None
    def _expr(self):
        left = self._additive()
        while self._peek()[0]=='OP' and self._peek()[1] in ('=','<>','<','>','<=','>='):
            op = self._eat()[1]; right = self._additive()
            left = self._compare(left, op, right)
        return left
    def _compare(self, a, op, b):
        return {'=':a==b, '<>':a!=b, '<':a<b, '>':a>b, '<=':a<=b, '>=':a>=b}[op]
    def _additive(self):
        left = self._term()
        while self._peek()[0]=='OP' and self._peek()[1] in ('+','-'):
            op = self._eat()[1]; right = self._term()
            left = self._num(left)+self._num(right) if op=='+' else self._num(left)-self._num(right)
        return left
    def _term(self):
        left = self._power()
        while self._peek()[0]=='OP' and self._peek()[1] in ('*','/'):
            op = self._eat()[1]; right = self._power()
            if op == '*': left = self._num(left) * self._num(right)
            else:
                r = self._num(right)
                if r == 0: raise FormulaError('#DIV/0!')
                left = self._num(left) / r
        return left
    def _power(self):
        left = self._factor()
        if self._peek()[0]=='OP' and self._peek()[1]=='^':
            self._eat(); right = self._factor()
            return self._num(left) ** self._num(right)
        return left
    def _factor(self):
        t = self._peek()
        if t[0]=='OP' and t[1]=='-': self._eat(); return -self._num(self._factor())
        if t[0]=='OP' and t[1]=='+': self._eat(); return self._num(self._factor())
        if t[0]=='NUM': return self._eat()[1]
        if t[0]=='STR': return self._eat()[1]
        if t[0]=='BOOL': return self._eat()[1]
        if t[0]=='OP' and t[1]=='(':
            self._eat(); v = self._expr()
            if not self._match('OP', ')'): raise FormulaError('#SYNTAX: missing )')
            return v
        if t[0]=='REF':
            r1 = self._eat()[1]
            if self._peek()[0]=='OP' and self._peek()[1]==':':
                self._eat()
                if self._peek()[0] != 'REF': raise FormulaError('#SYNTAX')
                r2 = self._eat()[1]
                return self._resolve_range(r1, r2)
            return self._resolve_cell(r1)
        if t[0]=='FUNC':
            name = self._eat()[1]
            if not self._match('OP', '('): raise FormulaError(f'#NAME?: {name}')
            args = []
            if not (self._peek()[0]=='OP' and self._peek()[1]==')'):
                args.append(self._expr())
                while self._peek()[0]=='OP' and self._peek()[1]==',':
                    self._eat(); args.append(self._expr())
            if not self._match('OP', ')'): raise FormulaError(f'#SYNTAX: missing ) in {name}')
            return self._call(name, args)
        raise FormulaError('#SYNTAX: unexpected token')
    def _resolve_cell(self, ref):
        r, c = a1_to_rc(ref); self.deps.add((r, c))
        v = self.get_cell(r, c)
        if v is None or v == '': return 0
        if isinstance(v, str):
            try: return float(v)
            except ValueError: return v
        return v
    def _resolve_range(self, r1, r2):
        rr1, cc1 = a1_to_rc(r1); rr2, cc2 = a1_to_rc(r2)
        rs, re_ = min(rr1,rr2), max(rr1,rr2)
        cs, ce = min(cc1,cc2), max(cc1,cc2)
        vals = []
        for r in range(rs, re_+1):
            for c in range(cs, ce+1):
                self.deps.add((r, c))
                v = self.get_cell(r, c)
                if v is None or v == '': continue
                if isinstance(v, (int, float)): vals.append(float(v))
                elif isinstance(v, str):
                    try: vals.append(float(v))
                    except ValueError: pass
        return vals
    def _call(self, name, args):
        if name == 'IF':
            if len(args) != 3: raise FormulaError('#VALUE!')
            return args[1] if args[0] else args[2]
        if name in ('CONCAT','CONCATENATE'):
            return ''.join(str(a) for a in args)
        if name not in FUNCTIONS: raise FormulaError(f'#NAME?: {name}')
        flat = []
        for a in args:
            if isinstance(a, list): flat.extend(a)
            elif isinstance(a, (int, float)): flat.append(float(a))
            elif isinstance(a, str):
                try: flat.append(float(a))
                except ValueError: pass
            elif isinstance(a, bool): flat.append(1.0 if a else 0.0)
        r = FUNCTIONS[name](flat)
        if isinstance(r, FormulaError): raise r
        return r
    def _num(self, v):
        if isinstance(v, bool): return 1 if v else 0
        if isinstance(v, (int, float)): return v
        if isinstance(v, str):
            try: return float(v)
            except ValueError: raise FormulaError('#VALUE!')
        if isinstance(v, list): raise FormulaError('#VALUE!')
        raise FormulaError('#VALUE!')

class Spreadsheet:
    def __init__(self, rows=20, cols=10):
        self.rows, self.cols = rows, cols
        self.raw, self.values = {}, {}
        self.deps_of, self.dependents = {}, {}
        self._history, self._future = [], []
        self._history_limit = 50
    def _snapshot(self):
        return {'raw': dict(self.raw), 'values': dict(self.values),
                'deps_of': {k: set(v) for k,v in self.deps_of.items()},
                'dependents': {k: set(v) for k,v in self.dependents.items()},
                'rows': self.rows, 'cols': self.cols}
    def _restore(self, s):
        self.raw = dict(s['raw']); self.values = dict(s['values'])
        self.deps_of = {k: set(v) for k,v in s['deps_of'].items()}
        self.dependents = {k: set(v) for k,v in s['dependents'].items()}
        self.rows, self.cols = s['rows'], s['cols']
    def _push(self):
        self._history.append(self._snapshot())
        if len(self._history) > self._history_limit: self._history.pop(0)
        self._future.clear()
    def undo(self):
        if not self._history: return False
        self._future.append(self._snapshot()); self._restore(self._history.pop()); return True
    def redo(self):
        if not self._future: return False
        self._history.append(self._snapshot()); self._restore(self._future.pop()); return True
    def set_cell(self, row, col, text, record_history=True):
        if record_history: self._push()
        key = (row, col)
        if key in self.deps_of:
            for d in self.deps_of[key]: self.dependents.get(d, set()).discard(key)
            del self.deps_of[key]
        if text is None or text == '':
            self.raw.pop(key, None); self.values.pop(key, None)
        else:
            self.raw[key] = text
        self._recalc(row, col)
        self._recalc_deps(key)
    def _recalc(self, row, col):
        key = (row, col); text = self.raw.get(key, '')
        if text == '': self.values.pop(key, None); return
        if isinstance(text, str) and text.startswith('='):
            ev = Evaluator(lambda r, c: self.values.get((r, c)))
            try:
                r = ev.evaluate(text[1:])
                if isinstance(r, list): r = r[0] if r else ''
                self.values[key] = r
                self.deps_of[key] = ev.deps
                for d in ev.deps: self.dependents.setdefault(d, set()).add(key)
            except FormulaError as e: self.values[key] = str(e)
            except Exception: self.values[key] = '#ERROR!'
        else:
            try: self.values[key] = float(text) if '.' in text or 'e' in text.lower() else int(text)
            except (ValueError, TypeError): self.values[key] = text
    def _recalc_deps(self, key):
        seen, queue = set(), list(self.dependents.get(key, set()))
        while queue:
            c = queue.pop(0)
            if c in seen: continue
            seen.add(c); self._recalc(*c)
            queue.extend(self.dependents.get(c, set()))
    def recalculate_all(self):
        for _ in range(5):
            changed = False
            for k in list(self.raw.keys()):
                before = self.values.get(k); self._recalc(*k)
                if self.values.get(k) != before: changed = True
            if not changed: break
    def get_raw(self, r, c): return self.raw.get((r, c), '')
    def get_value(self, r, c): return self.values.get((r, c), '')
    def get_display(self, r, c):
        v = self.values.get((r, c), '')
        if isinstance(v, float):
            if v.is_integer(): return str(int(v))
            return f'{v:g}'
        return str(v)
    def insert_row(self, at):
        self._push()
        self.raw = {((r+1 if r>=at else r), c): v for (r,c), v in self.raw.items()}
        self.rows += 1; self._rebuild()
    def delete_row(self, at):
        self._push()
        self.raw = {((r-1 if r>at else r), c): v for (r,c), v in self.raw.items() if r != at}
        self.rows = max(1, self.rows-1); self._rebuild()
    def insert_col(self, at):
        self._push()
        self.raw = {(r, (c+1 if c>=at else c)): v for (r,c), v in self.raw.items()}
        self.cols += 1; self._rebuild()
    def delete_col(self, at):
        self._push()
        self.raw = {(r, (c-1 if c>at else c)): v for (r,c), v in self.raw.items() if c != at}
        self.cols = max(1, self.cols-1); self._rebuild()
    def _rebuild(self):
        self.values.clear(); self.deps_of.clear(); self.dependents.clear()
        for k in list(self.raw.keys()): self._recalc(*k)
        self.recalculate_all()
    def load_from_2d(self, data):
        self.raw.clear(); self.values.clear()
        self.deps_of.clear(); self.dependents.clear()
        if data:
            self.rows = max(self.rows, len(data))
            self.cols = max(self.cols, max(len(r) for r in data))
        for r, row in enumerate(data):
            for c, val in enumerate(row):
                if val is None or val == '': continue
                self.set_cell(r, c, str(val), record_history=False)
        self.recalculate_all()
        self._history.clear(); self._future.clear()
    def save_csv(self, path):
        data = [['' for _ in range(self.cols)] for _ in range(self.rows)]
        for (r,c), v in self.values.items():
            if r < self.rows and c < self.cols: data[r][c] = v
        with open(path, 'w', newline='') as f: csv.writer(f).writerows(data)
    def load_csv(self, path):
        with open(path, newline='') as f: rows = list(csv.reader(f))
        self.load_from_2d(rows)
    def save_xlsx(self, path):
        from openpyxl import Workbook
        wb = Workbook(); ws = wb.active
        for (r,c), text in self.raw.items():
            ws.cell(row=r+1, column=c+1, value=text)
        wb.save(path)
    def load_xlsx(self, path):
        from openpyxl import load_workbook
        wb = load_workbook(path); ws = wb.active
        data = [list(row) for row in ws.iter_rows(values_only=True)]
        self.load_from_2d(data)

print('Engine loaded.')

Engine loaded.


In [9]:
# Cell 2: The ipywidgets UI
import ipywidgets as widgets
from IPython.display import display, clear_output

class MiniExcelUI:
    def __init__(self, rows=12, cols=8):
        self.sheet = Spreadsheet(rows, cols)
        self.cell_widgets = {}     # (r,c) -> Text widget
        self.selected = (0, 0)
        self._programmatic_update = False
        self._build()
    
    def _build(self):
        # Toolbar: formula bar + buttons
        self.cell_label = widgets.Label(value='A1', layout=widgets.Layout(width='50px'))
        self.formula_bar = widgets.Text(
            value='', placeholder='Enter value or formula (e.g. =SUM(A1:A5))',
            layout=widgets.Layout(width='500px'))
        self.formula_bar.observe(self._on_formula_bar_change, names='value')
        self.formula_bar.on_submit(self._on_formula_submit)

        btn = lambda d, t, cb: widgets.Button(description=d, tooltip=t,
                                              layout=widgets.Layout(width='90px')).on_click(cb) or _
        # Using factory helper gets awkward; construct directly instead.
        self.undo_btn = widgets.Button(description='Undo', icon='undo',
                                       layout=widgets.Layout(width='90px'))
        self.redo_btn = widgets.Button(description='Redo', icon='redo',
                                       layout=widgets.Layout(width='90px'))
        self.ins_row_btn = widgets.Button(description='+Row', tooltip='Insert row above selected',
                                          layout=widgets.Layout(width='70px'))
        self.del_row_btn = widgets.Button(description='-Row', tooltip='Delete selected row',
                                          layout=widgets.Layout(width='70px'))
        self.ins_col_btn = widgets.Button(description='+Col', tooltip='Insert column before selected',
                                          layout=widgets.Layout(width='70px'))
        self.del_col_btn = widgets.Button(description='-Col', tooltip='Delete selected column',
                                          layout=widgets.Layout(width='70px'))
        self.save_csv_btn = widgets.Button(description='Save CSV',
                                           layout=widgets.Layout(width='100px'))
        self.save_xlsx_btn = widgets.Button(description='Save XLSX',
                                            layout=widgets.Layout(width='100px'))
        self.clear_btn = widgets.Button(description='Clear All', button_style='warning',
                                        layout=widgets.Layout(width='100px'))
        self.path_input = widgets.Text(value='miniexcel_output', placeholder='filename (no extension)',
                                       layout=widgets.Layout(width='200px'))

        self.undo_btn.on_click(self._on_undo)
        self.redo_btn.on_click(self._on_redo)
        self.ins_row_btn.on_click(self._on_ins_row)
        self.del_row_btn.on_click(self._on_del_row)
        self.ins_col_btn.on_click(self._on_ins_col)
        self.del_col_btn.on_click(self._on_del_col)
        self.save_csv_btn.on_click(self._on_save_csv)
        self.save_xlsx_btn.on_click(self._on_save_xlsx)
        self.clear_btn.on_click(self._on_clear)

        toolbar1 = widgets.HBox([self.cell_label, self.formula_bar])
        toolbar2 = widgets.HBox([
            self.undo_btn, self.redo_btn,
            self.ins_row_btn, self.del_row_btn,
            self.ins_col_btn, self.del_col_btn,
            self.path_input, self.save_csv_btn, self.save_xlsx_btn, self.clear_btn,
        ])

        self.status = widgets.HTML(value='<span style="color:#888">Ready.</span>')
        self.grid_box = widgets.VBox([])
        self._render_grid()

        self.container = widgets.VBox([toolbar1, toolbar2, self.grid_box, self.status])

    def _render_grid(self):
        # Top header row: blank corner + column letters
        corner = widgets.Label(value='', layout=widgets.Layout(width='40px', border='1px solid #ccc'))
        headers = [corner]
        for c in range(self.sheet.cols):
            h = widgets.Label(value=col_letter(c),
                              layout=widgets.Layout(width='90px', border='1px solid #ccc'))
            h.add_class('mini-header')
            headers.append(h)
        header_row = widgets.HBox(headers)

        # Data rows
        rows = [header_row]
        self.cell_widgets.clear()
        for r in range(self.sheet.rows):
            row_widgets = [widgets.Label(value=str(r + 1),
                                          layout=widgets.Layout(width='40px', border='1px solid #ccc'))]
            for c in range(self.sheet.cols):
                w = widgets.Text(value=self.sheet.get_display(r, c),
                                 layout=widgets.Layout(width='90px'))
                w._coords = (r, c)
                w.observe(self._on_cell_change, names='value')
                self.cell_widgets[(r, c)] = w
                row_widgets.append(w)
            rows.append(widgets.HBox(row_widgets))
        self.grid_box.children = rows

    def _on_cell_change(self, change):
        if self._programmatic_update: return
        w = change['owner']; r, c = w._coords
        self.selected = (r, c)
        self.cell_label.value = rc_to_a1(r, c)
        self.sheet.set_cell(r, c, change['new'])
        self._refresh_all_cells()
        # After user finishes typing, show the raw in the formula bar.
        self._programmatic_update = True
        self.formula_bar.value = self.sheet.get_raw(r, c)
        self._programmatic_update = False
        self._set_status(f'Set {rc_to_a1(r,c)} = {self.sheet.get_raw(r,c)!r} -> {self.sheet.get_display(r,c)}')

    def _on_formula_bar_change(self, change):
        # Live update of the selected cell's raw as user types in the formula bar
        if self._programmatic_update: return
        r, c = self.selected
        self.sheet.set_cell(r, c, change['new'])
        self._refresh_all_cells()

    def _on_formula_submit(self, _):
        r, c = self.selected
        self._set_status(f'Formula committed for {rc_to_a1(r,c)}')

    def _refresh_all_cells(self):
        self._programmatic_update = True
        for (r, c), w in self.cell_widgets.items():
            new_display = self.sheet.get_display(r, c)
            if w.value != new_display and (r, c) != self.selected:
                # only overwrite cells the user isn't actively editing
                w.value = new_display
        self._programmatic_update = False

    def _on_undo(self, _):
        if self.sheet.undo():
            self._hard_refresh(); self._set_status('Undid last change.')
        else:
            self._set_status('Nothing to undo.')

    def _on_redo(self, _):
        if self.sheet.redo():
            self._hard_refresh(); self._set_status('Redid change.')
        else:
            self._set_status('Nothing to redo.')

    def _on_ins_row(self, _):
        r, _c = self.selected
        self.sheet.insert_row(r)
        self._render_grid(); self._hard_refresh()
        self._set_status(f'Inserted row at {r+1}.')

    def _on_del_row(self, _):
        r, _c = self.selected
        self.sheet.delete_row(r)
        self._render_grid(); self._hard_refresh()
        self._set_status(f'Deleted row {r+1}.')

    def _on_ins_col(self, _):
        _r, c = self.selected
        self.sheet.insert_col(c)
        self._render_grid(); self._hard_refresh()
        self._set_status(f'Inserted column at {col_letter(c)}.')

    def _on_del_col(self, _):
        _r, c = self.selected
        self.sheet.delete_col(c)
        self._render_grid(); self._hard_refresh()
        self._set_status(f'Deleted column {col_letter(c)}.')

    def _on_save_csv(self, _):
        path = (self.path_input.value or 'miniexcel_output') + '.csv'
        self.sheet.save_csv(path)
        self._set_status(f'Saved to {path}')

    def _on_save_xlsx(self, _):
        path = (self.path_input.value or 'miniexcel_output') + '.xlsx'
        self.sheet.save_xlsx(path)
        self._set_status(f'Saved to {path}')

    def _on_clear(self, _):
        self.sheet = Spreadsheet(self.sheet.rows, self.sheet.cols)
        self._render_grid(); self._hard_refresh()
        self._set_status('Cleared.')

    def _hard_refresh(self):
        self._programmatic_update = True
        for (r, c), w in self.cell_widgets.items():
            w.value = self.sheet.get_display(r, c)
        self._programmatic_update = False

    def _set_status(self, msg):
        self.status.value = f'<span style="color:#333">{msg}</span>'

    def load_sample(self):
        sample = [
            ['Product',  'Q1',   'Q2',   'Q3',   'Q4',   'Total',         'Avg'],
            ['Widgets',  '1200', '1450', '1600', '1800', '=SUM(B2:E2)',   '=AVERAGE(B2:E2)'],
            ['Gadgets',  '800',  '950',  '1100', '1300', '=SUM(B3:E3)',   '=AVERAGE(B3:E3)'],
            ['Gizmos',   '500',  '600',  '750',  '900',  '=SUM(B4:E4)',   '=AVERAGE(B4:E4)'],
            ['Doodads',  '300',  '350',  '400',  '450',  '=SUM(B5:E5)',   '=AVERAGE(B5:E5)'],
            ['Total',    '=SUM(B2:B5)', '=SUM(C2:C5)', '=SUM(D2:D5)', '=SUM(E2:E5)',
                         '=SUM(F2:F5)', '=AVERAGE(G2:G5)'],
            [],
            ['Stats',    'Value'],
            ['Best quarter Q4 total', '=E6'],
            ['Growth Q1->Q4 (Widgets)', '=(E2-B2)/B2'],
            ['High performer?', '=IF(F2>5000, "Yes", "No")'],
        ]
        self.sheet.load_from_2d(sample)
        self._render_grid(); self._hard_refresh()
        self._set_status('Sample data loaded. Try editing B2 and watch totals recalc.')

    def show(self):
        display(self.container)

app = MiniExcelUI(rows=14, cols=8)
app.load_sample()
app.show()

/var/folders/zh/k0v6dl515cv8bvrzx9j1cpcm0000gn/T/ipykernel_64826/3646899743.py:20: DeprecationWarning: on_submit is deprecated. Instead, set the .continuous_update attribute to False and observe the value changing with: mywidget.observe(callback, 'value').
  self.formula_bar.on_submit(self._on_formula_submit)


AttributeError: 'Text' object has no attribute 'on_displayed'

## How to use

- **Edit a cell**: click a cell and type. Press Tab or Enter to commit.
- **Formulas**: start with `=`. Examples: `=A1+B1`, `=SUM(A1:A5)`, `=IF(A1>10, "big", "small")`.
- **Structural edits**: click a cell, then use `+Row`, `-Row`, `+Col`, `-Col`.
- **Undo / redo**: the toolbar buttons.
- **Save**: pick a filename, then click Save CSV or Save XLSX. Files land in your notebook's working directory.
- **Clear All**: wipes the sheet (undo-able).

## Supported functions
`SUM`, `AVERAGE`, `MIN`, `MAX`, `COUNT`, `PRODUCT`, `ABS`, `ROUND`, `IF`, `CONCAT` / `CONCATENATE`

Operators: `+ - * / ^`, comparisons `= <> < > <= >=`, parentheses, unary `-`.

## Known limitations
- Cell formulas do NOT auto-adjust when you insert/delete rows or columns (references are not rewritten).
- No multi-cell selection, copy/paste, or column resize yet.
- Formula bar currently updates the cell live on every keystroke; commit with Enter.
- Font, color, and cell formatting are not persisted.